In [ ]:
#| hide
from vishalakshi import *
from fastcore.all import *

# concepts

> what retrieval returns, which encoder wrote it, and why one file holds several vector spaces

The [README](index.html) shows what vishalakshi *does*. This page is the other half: the handful of
ideas you need when a default stops being the right one — when a corpus wants a different embedder,
when an answer should come from a bigger model, or when two shelves should not be compared.

In [ ]:
from tempfile import mkdtemp
from vishalakshi import Vault

v = Vault(Path(mkdtemp())/'vault.db')
v.note('federate fuses the legs by rank because they share no vector space: the vault embeds '
       'prose, kosha embeds identifiers, ripgrep embeds nothing.', tags=['retrieval'])
v.add('# Late chunking\n\n## Method\n\nEmbed the whole document, then pool per chunk, so a chunk '
      'keeps the context around it.\n\n## Results\n\nEvaluated on BEIR.', 'Late chunking', kind='note')
v.connect()
v.stats()

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'docs': 2,
 'nodes': 6,
 'chunks': 3,
 'encoder': 'model2vec',
 'entities': 21,
 'path': '/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmph5ysy_7x/vault.db',
 'by_kind': {'note': 2}}

## What retrieval actually returns

`v.context(q)` is the primitive underneath `ask`. It returns whole **sections** — the unit worth
reading — not fragments:

In [ ]:
c = v.context('why are rankings fused instead of distances?', sections=4, related=4)
for s in c.results: print(f'{len(s.text):5}  {s.breadcrumb}')

  141  federate fuses the legs by rank because they share no vector space: the vault em
   86  Late chunking › Method
   18  Late chunking › Results
  234  repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/code.py:57
  154  grep › index.ipynb:161
  600  repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/extract.py:179
   89  grep › 07_concepts.ipynb:101


In [ ]:
for s in c.related: print(f'{s.via:6}  {s.breadcrumb}')

`results` are the sections that answer the question. `related` are sections reached *by
association* — along the entity graph (`via='graph'`) or by embedding similarity (`via='vector'`) —
which is how you find the thing you did not know to search for.

In [ ]:
c.encoder

'minishlab/potion-multilingual-128M (256d, float16, model2vec)'

`c.encoder` always says which embedder answered, because degrading from real semantics to
hashing changes what the results mean.

## Models and backends

`ask` goes through rishi, and *only* through rishi: there is no model table here and nothing to
resolve. rishi picks the backend from the **shape** of the model id, so a bare marketing name
(`gemma-3-4b-it-int4`) matches nothing and is rejected rather than guessed at. Name a real id, name
one of rishi's own — `rishi.litert.gemma4_e2b`, `rishi.mlx.qwen3_4b` — or prefix the runtime.

| runtime | what it is | model ids look like | needs |
|---|---|---|---|
| `litert` | Google LiteRT, CPU — **the default** | `litert-community/…`, `*.litertlm` | nothing; runs anywhere |
| `mlx` | Apple silicon | `mlx-community/…` | macOS on ARM |
| `llama` | llama.cpp, any GGUF | `*-GGUF`, `*.gguf` | `pip install 'rishi[llama]'` |
| `remote` | hosted, via fastllm | `gpt-5.6-luna`, `gemini-…` | an API key |

In [ ]:
from rishi.core import Chat, infer_runtime
from vishalakshi.ask import dflt_model

# rishi's backend modules name their models — `rishi.litert.gemma4_e2b` is this id — but importing
# one imports its backend, so a page about ids talks about them as ids
gemma4_e2b = 'litert-community/gemma-4-E2B-it-litert-lm'
dflt_model, gemma4_e2b     # $VISHALAKSHI_MODEL if set, else rishi's own small local default

In [ ]:
{m: infer_runtime(m) for m in (gemma4_e2b, 'mlx-community/Qwen3-4B-4bit', 'my-local.gguf',
                               'gpt-5.6-luna', 'gemma-3-4b-it-int4')}

In [ ]:
#| eval: false
v.ask('why are rankings fused?', model=gemma4_e2b)             # an id: rishi knows what runs it
v.ask('why are rankings fused?', model='llama/my-local.gguf')  # a prefix when the id cannot say
v.ask('why are rankings fused?', model='mlx-community/Qwen3-4B-4bit',
      chat_kw=dict(temp=0, think=True))          # the rest of rishi's constructor

In [ ]:
from fastcore.test import test_fail
# `model=` plus `chat_kw=` is the whole model API: anything rishi's constructor takes, and rishi's
# own error when the id says nothing about which backend should run it
test_fail(lambda: Chat('gemma-3-4b-it-int4'), contains='backend')

A reasoning model's `<think>` block is split off into `r.thinking` rather than left in
`r.answer` — it names sections it then discards, and citations are read off the answer.

The vault never needs the network to *retrieve*. Only answering with a hosted model does.

## Encoders

The vault wants a real embedder and will fetch a small model2vec one by default. Where that is
impossible — an air-gapped box, a blocked registry — it degrades to litesearch's deterministic
`hash_embed` rather than failing, and says so in `stats()['encoder']` and on every `context()`
result. Retrieval still works; it is lexical rather than semantic, and you should know which you are
getting.

In [ ]:
Vault(':memory:', offline=True).enc.note      # never attempt a download — also the CI default

'char-n-gram hashing (256d) — lexical only; pass encoder= or restore network access for real semantics'

In [ ]:
#| eval: false
Vault(encoder='minishlab/potion-science-32M')   # pick a different one

Vault('/Users/71293/.vishalakshi/vault.db': 0 docs, 0 chunks, 0 entities, encoder=model2vec)

Both encoders are stored at float16, which is litesearch's own default width. That is not a
size optimisation: `Database.context` does not thread a `dtype=` down to its section search, so a
float32 store is read back there as float16 — the vectors survive, the *distances* do not, and
retrieval silently degrades to keyword ranking.

The vault takes any encoder [litesearch](https://github.com/vedicreader/litesearch) ships, by name:

In [ ]:
from vishalakshi.core import ENCODERS, enc_spec

{k: (m if isinstance(m, str) else m['model']) for k, m in ENCODERS.items()}

{'default': 'minishlab/potion-multilingual-128M',
 'multilingual': 'minishlab/potion-multilingual-128M',
 'retrieval': 'minishlab/potion-retrieval-32M',
 'science': 'minishlab/potion-science-32M',
 'code': 'minishlab/potion-code-16M-v2',
 'sanskritgemma': 'karthikrajgopal/sanskritgemma-256',
 'gemma': 'onnx-community/embeddinggemma-300m-ONNX',
 'bge-micro': 'TaylorAI/bge-micro-v2',
 'modernbert': 'nomic-ai/modernbert-embed-base',
 'nomic': 'nomic-ai/nomic-embed-text-v1.5'}

Two kinds, and the difference is a real trade rather than a detail. A static model is a lookup
table: milliseconds per document, no GPU, no batching. An ONNX transformer actually reads word order
and costs perhaps a hundred times more per chunk. Ingest is where that bill lands, so the default
stays static and the choice stays yours — `science` for papers, `gemma` or `bge-micro` when fidelity
is worth the wait, `code` for identifiers.

In [ ]:
enc_spec('science'), enc_spec('bge-micro')[1], enc_spec('gemma')[0]['model']

(('minishlab/potion-science-32M', 'static'),
 'onnx',
 'onnx-community/embeddinggemma-300m-ONNX')

What you cannot do is mix them *inside* one index. One ANN index is one vector space; two models'
vectors in it are compared as bytes and the distances mean nothing. So a corpus that wants a
different embedder gets a **shelf** — same file, own store, own index — and the shelves are read
together rather than merged. `SHELVES` is the layout, and it is meant to be edited:

In [ ]:
from vishalakshi.core import SHELVES, KIND_SHELF

SHELVES, KIND_SHELF

({'store': 'default',
  'papers': 'science',
  'sanskrit': 'gemma',
  'sanskrit-fast': 'default',
  'code': 'code',
  'data': 'retrieval'},
 {'arxiv': 'papers', 'sanskrit': 'sanskrit'})

In [ ]:
papers = v.shelf('papers')        # no encoder named: the registry knows it is a science model
papers.add('# Late chunking\n\nWe evaluate contextual chunk embeddings on BEIR.', 'a paper')

[(s['store'], s['encoder'], s['docs']) for s in v.shelves()]

[('store', 'minishlab/potion-multilingual-128M', 2),
 ('papers', 'minishlab/potion-science-32M', 1)]

`KIND_SHELF` is where acquisition routes, and it is deliberately one entry. `v.grab('2404.12345')`
detects an arXiv id and files it on the science shelf; so does `v.arxiv(id)`, because the method name
fixes the kind. Nothing else routes, and that restraint is the design: `find` and `sections` read one
shelf, so a write that quietly moves is a read that quietly comes back empty. An explicit shelf is
never overruled either — `shelf('sanskrit').arxiv(id)` files that paper in Sanskrit.

In [ ]:
v.route('arxiv').name, v.route('web').name

('papers', 'store')

Which leaves the interesting half. A PDF is as likely an invoice as a paper, and nothing at the door
can tell — only `categorize` can, and only once the document is already somewhere. So the cheap route
runs at ingest, and `reshelf` runs afterwards, on what the document turned out to be:

In [ ]:
v.add('# Attention is all you need\n\n## Abstract\n\nWe propose the Transformer.\n\n'
      '## Introduction\n\nRelated work [1] et al.\n\n## References\n\ndoi:10.1/x',
      'attention', source='/inbox/attention.pdf')

r = v.reshelf('/inbox/attention.pdf', llm='never')
r.doctype, r.was, r.store, r.moved

('paper', 'store', 'papers', True)

In [ ]:
v.doc('/inbox/attention.pdf'), v.shelf('papers').doc('/inbox/attention.pdf')['title']

(None, 'attention')

A move is a re-ingest — the other shelf has a different encoder, so the vectors have to be made again
— and it carries the reassembled document rather than the original file, so a PDF's page boundaries
do not survive it. Re-`grab` the source when the pages matter. `DOCTYPE_SHELF` is the mapping, and
like `SHELVES` it is meant to be edited.

Reading spans the library two ways. `federate` fuses the shelves by rank alongside kosha and
ripgrep — every shelf in the file, by default:

In [ ]:
v.federate('contextual chunk embeddings', repo=False, grep=False).legs

{'prose': 3, 'shelf:papers': 4}

…and `context` — so `ask` — appends a couple of sections from each *other* shelf to what it
retrieved here, tagged with where they came from. A shelf's scores are not comparable with this
shelf's, so they arrive as extra sections for the model to weigh rather than merged into one
ranking, and `read(node_id, store=…)` opens any of them:

In [ ]:
e = v.elsewhere('contextual chunk embeddings')
[(r.store, r.breadcrumb) for r in e]

[('papers', 'papers › a paper'),
 ('papers', 'papers › attention › Attention is all you need › Abstract')]

In [ ]:
v.read(e[0].node_id, store=e[0].store)['text'][:80]

'# Late chunking\n\nWe evaluate contextual chunk embeddings on BEIR.'

A shelf records the encoder that wrote it, so it reopens with the right one and says so loudly when
it does not. That warning is the point of the registry: litesearch already warns when the stored
vector *width* disagrees, but when the widths match and the models differ — two 256-dim static
models, or a real encoder quietly replaced by the hashing fallback after a failed download — nothing
raises and every distance is computed across two different spaces.